# NimbusPay ticket triage: LoRA fine-tune of Qwen2.5-3B-Instruct

Pipeline: setup, data audit and cleaning, sequence length analysis, baselines, training, controlled experiments, error analysis, final model and test predictions.

Runs on a free Colab T4. Every run saves its adapter, logs and predictions to Google Drive under `MyDrive/nimbus_triage/runs/`.

## 0. Setup

Pinned library versions, seeds, paths and shared helpers. On Colab the data is read from `MyDrive/nimbus_triage/candidate_pack/`; locally it is read from `./candidate_pack/`.

In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

IN_COLAB = "google.colab" in sys.modules

PINS = {
    "transformers": "5.22.1",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
}
EXPECTED_TORCH = "2.14.1"


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


if IN_COLAB:
    missing = [f"{p}=={v}" for p, v in PINS.items() if installed(p) != v]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

print("colab:", IN_COLAB)
print(f"torch=={installed('torch')}", "" if (installed("torch") or "").startswith(EXPECTED_TORCH) else f"(expected {EXPECTED_TORCH})")
for pkg, want in PINS.items():
    have = installed(pkg)
    print(f"{pkg}=={have}", "" if have == want else f"(expected {want})")

In [ ]:
import csv
import json
import random
import time
import importlib.util
from datetime import datetime
from pathlib import Path

import numpy as np

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/nimbus_triage")
else:
    ROOT = Path.cwd()

PACK = ROOT / "candidate_pack"
DATA = PACK / "data"
RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SEED = 42
MAX_NEW_TOKENS = 200

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
CATEGORIES = ["payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"]
PRIORITIES = ["P1", "P2", "P3"]
CHANNELS = ["upi", "card", "netbanking", "wallet"]
LANGUAGES = ["en", "hi-en"]
SOURCES = ["app-chat", "email", "twitter", "ivr-transcript"]
TXN_FIELDS = ["amount_paise", "txn_date", "txn_id", "channel"]
NON_TXN_CATEGORIES = ["account_access", "kyc", "other"]

for p in [DATA / "train.jsonl", DATA / "dev.jsonl", DATA / "test_inputs.jsonl", PACK / "score.py", PACK / "SCHEMA.md"]:
    assert p.exists(), f"missing {p}"

print("root:", ROOT)
print("runs:", RUNS)

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def system_text(row):
    return row["messages"][0]["content"]


def user_text(row):
    return row["messages"][1]["content"]


def label_text(row):
    return row["messages"][2]["content"]


def ticket_body(row):
    return user_text(row).split("\n\n", 1)[-1]


def received_date(row):
    return datetime.strptime(user_text(row).split("\n", 1)[0].split()[1], "%Y-%m-%d").date()


def ticket_source(row):
    return user_text(row).split("\n")[1].split(": ", 1)[1].strip()


def canonical(label):
    return json.dumps({k: label[k] for k in KEYS}, ensure_ascii=False)


_spec = importlib.util.spec_from_file_location("nimbus_score", PACK / "score.py")
scorer = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(scorer)


def evaluate(gold_rows, pred_rows):
    return scorer.score(gold_rows, pred_rows)


def show_scores(res, prefixes=None):
    print(f"{'group':28} {'n':>5} {'valid':>7} {'exact':>7} {'fields':>7}")
    for name in sorted(res, key=lambda x: (x != "all", x)):
        if prefixes is not None and name != "all" and not name.startswith(tuple(prefixes)):
            continue
        r = res[name]
        print(f"{name:28} {r['n']:5d} {r['json_valid']:7.1%} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
    print("per-field accuracy:", {k: round(v, 3) for k, v in res["all"]["fields"].items()})


EXPERIMENTS_CSV = RUNS / "experiments.csv"
EXPERIMENT_COLUMNS = ["run", "change", "train_rows", "dev_exact", "dev_field_acc", "dev_json_valid",
                      "peak_gpu_gb", "train_minutes", "trainable_params", "timestamp"]


def log_experiment(record):
    record = {**record, "timestamp": datetime.now().isoformat(timespec="seconds")}
    new = not EXPERIMENTS_CSV.exists()
    with open(EXPERIMENTS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=EXPERIMENT_COLUMNS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow(record)
    return record


train_raw = load_jsonl(DATA / "train.jsonl")
dev = load_jsonl(DATA / "dev.jsonl")
test = load_jsonl(DATA / "test_inputs.jsonl")
print(len(train_raw), len(dev), len(test))
print(user_text(dev[0]))
print(label_text(dev[0]))

In [ ]:
import torch
import transformers


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)


set_seed()

print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    NATIVE_BF16 = props.major >= 8
    print("gpu:", props.name, f"{props.total_memory / 1e9:.1f} GB", f"| compute capability {props.major}.{props.minor}",
          "| native bf16:", NATIVE_BF16, "| training dtype:", "bf16" if NATIVE_BF16 else "fp16")
else:
    print("gpu: none")

## 1. Part A: data audit

The training labels came out of a noisy pipeline. The audit has three layers:

1. **Structure**: the label must parse as JSON, have the 8 spec keys, use the right types and formats, not repeat dev or test tickets, and not be a duplicate or a junk ticket.
2. **Rules**: a reference labeller written only from `SCHEMA.md` and the phrasing seen in `train.jsonl` reads each ticket and works out the fields the spec defines mechanically (priority, needs_human, txn_id, amount, date, channel). Every disagreement with the given label is a finding.
3. **Consistency**: tickets that open with the same issue sentence should share a category and a language. Rows that go against their group are flagged.

The reference labeller is only an audit tool. It is never used at inference. It was built from train only. Dev is used once at the end to measure how accurate the labeller is, not to tune it.

In [ ]:
import re
from collections import Counter, defaultdict

import pandas as pd

FIXED_SYSTEM = system_text(dev[0])
findings = defaultdict(set)


def flag(code, rid):
    findings[code].add(rid)


def parse_label(text):
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def norm_text(s):
    return re.sub(r"\s+", " ", s).strip().lower()


ISO_DATE = re.compile(r"\d{4}-\d{2}-\d{2}")
DMY_DATE = re.compile(r"(\d{2})/(\d{2})/(\d{4})")
TXN_ID = re.compile(r"NP\d{10}")

labels = {}
for r in train_raw:
    rid = r["id"]
    if [m["role"] for m in r["messages"]] != ["system", "user", "assistant"]:
        flag("bad_roles", rid)
    if system_text(r) != FIXED_SYSTEM:
        flag("bad_system_prompt", rid)
    y = parse_label(label_text(r))
    if y is None:
        flag("label_not_json", rid)
        continue
    labels[rid] = y
    if "prio" in y:
        flag("schema_v1_prio_key", rid)
    elif set(y) != set(KEYS):
        flag("wrong_keys", rid)
    if y.get("category") not in CATEGORIES:
        flag("bad_category_value", rid)
    if "priority" in y and y["priority"] not in PRIORITIES:
        flag("bad_priority_value", rid)
    a = y.get("amount_paise")
    if a is not None and type(a) is not int:
        flag("amount_wrong_type", rid)
    d = y.get("txn_date")
    if d is not None:
        if isinstance(d, str) and DMY_DATE.fullmatch(d):
            flag("date_dd_mm_yyyy", rid)
        elif not (isinstance(d, str) and ISO_DATE.fullmatch(d)):
            flag("date_bad_format", rid)
        elif d > str(received_date(r)):
            flag("date_after_received", rid)
    t = y.get("txn_id")
    if t is not None and not (isinstance(t, str) and TXN_ID.fullmatch(t)):
        flag("txn_id_not_normalized", rid)
    if y.get("channel") not in [None, *CHANNELS]:
        flag("bad_channel_value", rid)
    if y.get("language") not in LANGUAGES:
        flag("bad_language_value", rid)
    if type(y.get("needs_human")) is not bool:
        flag("needs_human_wrong_type", rid)
    if y.get("category") in NON_TXN_CATEGORIES and any(y.get(k) is not None for k in TXN_FIELDS):
        flag("txn_fields_on_non_txn_category", rid)

dev_users = {norm_text(user_text(r)) for r in dev}
test_users = {norm_text(user_text(r)) for r in test}
dev_bodies = {norm_text(ticket_body(r)) for r in dev}
test_bodies = {norm_text(ticket_body(r)) for r in test}
for r in train_raw:
    u, b = norm_text(user_text(r)), norm_text(ticket_body(r))
    if u in dev_users:
        flag("leak_dev_exact", r["id"])
    elif b in dev_bodies:
        flag("leak_dev_same_body", r["id"])
    if u in test_users:
        flag("leak_test_exact", r["id"])
    elif b in test_bodies:
        flag("leak_test_same_body", r["id"])

groups = defaultdict(list)
for r in train_raw:
    groups[norm_text(user_text(r))].append(r)
dup_groups = [g for g in groups.values() if len(g) > 1]
for g in dup_groups:
    distinct = {label_text(r) for r in g}
    for r in g[1:]:
        flag("duplicate_same_label" if len(distinct) == 1 else "duplicate_conflicting_label", r["id"])

JUNK = re.compile(r"(test|asdf|qwerty|hello|hi|\d|\W|_)*", re.I)
for r in train_raw:
    body = re.sub(r"^(\[IVR transcript\]|@NimbusPayCare)", "", ticket_body(r).strip()).strip()
    if len(body) < 20 and JUNK.fullmatch(body):
        flag("junk_ticket", r["id"])

print(f"train rows {len(train_raw)}, parsed labels {len(labels)}, duplicate groups {len(dup_groups)}")
structure_table = pd.DataFrame(
    [(k, len(v), ", ".join(sorted(v)[:4])) for k, v in sorted(findings.items(), key=lambda x: -len(x[1]))],
    columns=["finding", "rows", "examples"],
)
structure_table